# Mean, Median, Mode of Fund Returns and AuM

"The average Swiss equity fund manages CHF 400 million" and "the typical fund manages CHF 150 million" can both be true. The first is a
mean, the second a median, and the gap between them is a signature of a skewed distribution. Fund sizes, stock returns over long
horizons and wealth are all strongly right-skewed: a few giants pull the mean far above the typical value. This notebook defines the
three measures of centre, shows what each one minimizes, and applies them to fund assets under management (AuM), peer-group returns and
long-horizon stock returns.

## 0. Math link

- [Mean / expected value](https://github.com/dnlwrthstr/LearningProbability/blob/main/coursera/02_discribing_distribution/01_mean_expected_value.ipynb):
  the balance point of a distribution, $E[X] = \sum_x x\,p(x)$ or $\int x f(x)\,dx$.
- [Median and mode](https://github.com/dnlwrthstr/LearningProbability/blob/main/coursera/02_discribing_distribution/02_median_and_mode.ipynb):
  the 50% point of the cumulative distribution, and the peak of the density.
- Log returns and volatility drag from [00/01](../00_returns_and_compounding/01_simple_vs_log_returns.ipynb), which explain why
  long-horizon wealth is skewed.

## 1. Introduction

A fund selector screens 2,000 funds. The asset-weighted average return of the peer group, the equal-weighted average, the median fund
and the most common return are four different numbers, and marketing material picks whichever looks best. Knowing which measure answers
which question is the first step in describing returns honestly.

After this notebook you can:

- compute mean, median and mode for discrete and continuous data;
- explain which loss each measure minimizes (squared, absolute, 0–1);
- predict the ordering of mean, median and mode for right-skewed data such as AuM;
- distinguish equal-weighted and asset-weighted average returns;
- explain why most single stocks end below the average stock over long horizons.

## 2. Theory

### Three measures of centre

For a sample $x_1, \dots, x_n$:

- **mean** $\bar x = \frac1n\sum_i x_i$; for a distribution, $E[X]$;
- **median** $m$: half the observations below, half above; for a distribution, $F(m) = \tfrac12$;
- **mode**: the most frequent value; for a continuous distribution, the maximum of the density $f$.

### What each one minimizes

- The mean minimizes the squared error: $\frac{d}{dc}\sum_i (x_i - c)^2 = -2\sum_i (x_i - c) = 0 \Rightarrow c = \bar x$.
- The median minimizes the absolute error: $\frac{d}{dc}\sum_i |x_i - c| = \#\{x_i < c\} - \#\{x_i > c\}$, which changes sign at the median.
- The mode maximizes the chance of being exactly right (0–1 loss).

The mean reacts to every observation in proportion to its distance; the median only to which side it falls on. One outlier can move the
mean arbitrarily far but moves the median at most to a neighbouring value: the median is **robust**.

### Skewness and the ordering

For a right-skewed distribution (long right tail) typically

$$
\text{mode} < \text{median} < \text{mean} .
$$

A lognormal variable $X = e^{Y}$ with $Y \sim N(\mu, \sigma^2)$ has

$$
\text{mode} = e^{\mu - \sigma^2}, \qquad \text{median} = e^{\mu}, \qquad \text{mean} = e^{\mu + \sigma^2/2} .
$$

The ratio mean/median is $e^{\sigma^2/2}$: with $\sigma = 1.5$ (typical for fund sizes) the mean is 3.1 times the median.

### Equal-weighted and asset-weighted averages

The return of "the average fund" is $\frac1n\sum_i r_i$. The return of "the average franc invested" is

$$
\bar r_{AW} = \sum_i \frac{A_i}{\sum_j A_j}\, r_i ,
$$

weighting each fund by its AuM $A_i$. The two differ when large and small funds perform differently.

### Long-horizon wealth is lognormal

If a stock's log returns are independent with mean $\mu$ and volatility $\sigma$ per year, its wealth after $T$ years is lognormal
with median $e^{\mu T}$ and mean $e^{(\mu + \sigma^2/2)T}$. The probability that a single stock ends below the mean is
$\Phi\big(\tfrac{\sigma\sqrt{T}}{2}\big)$, which tends to 1 as $T$ grows: most stocks underperform the average stock.

## 3. Worked Examples

### Example 1: five funds

AuM (CHF million): 40, 80, 120, 300, 2,460. Mean 600, median 120. Remove the largest fund: the mean falls to 135, the median to 100.

### Example 2: which loss?

A fund selector must forecast next year's return of a fund from its peers' returns −2%, 3%, 4%, 5%, 20%. To minimize the squared error,
forecast the mean, 6%; to minimize the absolute error, the median, 4%. The 20% outlier pulls the mean but not the median.

### Example 3: equal- and asset-weighted

Two funds: CHF 900 million returning 2% and CHF 100 million returning 12%. Equal-weighted average 7%; asset-weighted
$0.9 \times 2\% + 0.1 \times 12\% = 3\%$. The average fund did well; the average franc did not.

### Example 4: a lognormal stock

Annual log-return mean 6%, volatility 40%, 20 years. Median wealth $e^{1.2} = 3.32$, mean wealth $e^{(0.06 + 0.08) \times 20} = e^{2.8} = 16.4$.
The probability of ending below the mean is $\Phi(0.4\sqrt{20}/2) = \Phi(0.894) = 81\%$.

## 4. Finance Application

### Where it matters

- **Industry statistics.** Fund-size and assets-per-client figures are skewed; medians describe the typical case, means the total.
- **Peer-group rankings.** Fund ratings compare a fund with the median of its category; quartile rankings are built on quantiles,
  not means.
- **Index construction.** Cap-weighted indices report the asset-weighted return; equal-weighted indices the average company.
- **Stock picking.** Long-horizon index returns are driven by a small fraction of stocks; the median stock underperforms the index
  (Bessembinder, 2018, *Do stocks outperform Treasury bills?*).
- **Wealth and pension statistics.** Average pension assets exceed the typical member's assets by a wide margin.

### The usual misreadings

- "The average fund is CHF 600 million, so a CHF 200 million fund is small." It may be above the median.
- "The peer group returned 7%." Equal- or asset-weighted? The difference can be several percentage points.
- "The average stock returned 10% a year, so a typical stock will too." The typical (median) stock compounds well below the mean.
- "The mode is the most likely return." For continuous returns the mode depends on how you bin the data.

## 5. Python Examples

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

### A synthetic universe of 2,000 funds

AuM is lognormal with median CHF 150 million and log-standard deviation 1.5. Annual returns (decimals) are normal around 5% with 4%
dispersion, plus a small size effect: larger funds return a little less (higher costs of trading large positions are one story).

In [ ]:
n_funds = 2000
log_aum = rng.normal(np.log(150), 1.5, n_funds)
aum = np.exp(log_aum)                                   # CHF million
size_effect = -0.005 * (log_aum - log_aum.mean())       # -0.5 points per unit of log AuM
returns = 0.05 + size_effect + rng.normal(0, 0.04, n_funds)
funds = pd.DataFrame({"aum_chf_m": aum, "return": returns})

def mode_from_histogram(x, width):
    # centre of the most populated bin of the given width
    edges = np.arange(np.min(x), np.max(x) + width, width)
    counts, edges = np.histogram(x, bins=edges)
    k = np.argmax(counts)
    return (edges[k] + edges[k + 1]) / 2

def mode_from_kde(x, grid_size=2000):
    kde = stats.gaussian_kde(x)
    grid = np.linspace(np.min(x), np.max(x), grid_size)
    return grid[np.argmax(kde(grid))]

summary = pd.DataFrame({
    "mean": [funds.aum_chf_m.mean(), funds["return"].mean()],
    "median": [funds.aum_chf_m.median(), funds["return"].median()],
    "mode": [mode_from_histogram(aum, 10), mode_from_kde(returns)],
}, index=["AuM (CHF million)", "annual return (decimal)"])
summary

For AuM the mean is almost three times the median, close to the lognormal prediction $e^{1.5^2/2} = 3.1$, and the mode (the most
populated CHF 10 million bin) is far below both, near the theoretical $e^{\mu - \sigma^2} \approx$ CHF 16 million. For returns the mean and
median coincide at 5.1%: the distribution is close to symmetric. The mode, estimated from a smoothed density (KDE), lands about a point
higher; for continuous data it is the least stable of the three.

In [ ]:
mu, sigma = np.log(150), 1.5
print(f"lognormal theory: mode {np.exp(mu - sigma**2):,.1f}, median {np.exp(mu):,.1f}, mean {np.exp(mu + sigma**2 / 2):,.1f} CHF million")
share_below_mean = (aum < aum.mean()).mean()
print(f"share of funds smaller than the mean fund: {share_below_mean:.1%}")
top_share = np.sort(aum)[::-1][: n_funds // 10].sum() / aum.sum()
print(f"share of total AuM in the largest 10% of funds: {top_share:.1%}")

### Equal- and asset-weighted returns

In [ ]:
ew = funds["return"].mean()
aw = np.average(funds["return"], weights=funds.aum_chf_m)
print(f"equal-weighted mean return: {ew:.3%}")
print(f"asset-weighted mean return: {aw:.3%}")
print(f"median fund return:         {funds['return'].median():.3%}")

Large funds return a little less in this universe, so the average franc earned less than the average fund. Which number is "the
peer-group return" depends on the question: the experience of investors (asset-weighted) or of fund choices (equal-weighted).

### Robustness: one outlier

In [ ]:
peer = funds["return"].to_numpy()[:30].copy()
print(f"30 funds:            mean {peer.mean():.3%}, median {np.median(peer):.3%}")
peer_out = peer.copy()
peer_out[0] = 3.0                       # one fund with +300% (a data error, or a lucky crypto fund)
print(f"with a +300% outlier: mean {peer_out.mean():.3%}, median {np.median(peer_out):.3%}")

One extreme value moves the mean of 30 funds by about 10 percentage points and the median hardly at all.

### Long-horizon stock returns: most stocks lose to the average stock

In [ ]:
n_stocks, years = 1000, 20
mu_log, vol = 0.06, 0.40                # annual log-return mean and volatility
log_wealth = rng.normal(mu_log * years, vol * np.sqrt(years), n_stocks)
wealth = np.exp(log_wealth)             # CHF per CHF 1 invested 20 years ago

print(f"mean wealth   {wealth.mean():7.2f}  (theory {np.exp((mu_log + vol**2 / 2) * years):.2f})")
print(f"median wealth {np.median(wealth):7.2f}  (theory {np.exp(mu_log * years):.2f})")
print(f"share of stocks below the mean: {(wealth < wealth.mean()).mean():.1%}  "
      f"(theory {stats.norm.cdf(vol * np.sqrt(years) / 2):.1%})")
print(f"share of stocks that lost money: {(wealth < 1).mean():.1%}")
top = np.sort(wealth)[::-1]
print(f"share of total end wealth from the best 4% of stocks: {top[: n_stocks * 4 // 100].sum() / wealth.sum():.1%}")

An equal-weighted portfolio of all 1,000 stocks ends with the mean wealth; the typical stock ends with the median, several times less.
A handful of extreme winners produce a large share of the total. The sample mean itself is noisy here: it depends heavily on those few
stocks.

## 6. Visualizations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].hist(aum[aum < 3000], bins=np.arange(0, 3000, 25), color="C0", alpha=0.7)
axes[0].set_xlim(0, 3000)
axes[0].set_title("Fund AuM, linear scale (cut at CHF 3 bn)")
axes[0].set_xlabel("AuM (CHF million)")
axes[0].set_ylabel("number of funds")

axes[1].hist(np.log10(aum), bins=50, color="C0", alpha=0.7)
axes[1].set_title("Fund AuM, log scale")
axes[1].set_xlabel("log10 AuM (CHF million)")
axes[1].set_ylabel("number of funds")

for ax, transform in [(axes[0], lambda v: v), (axes[1], np.log10)]:
    ax.axvline(transform(aum.mean()), color="C3", lw=2, label=f"mean {aum.mean():,.0f}")
    ax.axvline(transform(np.median(aum)), color="C2", lw=2, ls="--", label=f"median {np.median(aum):,.0f}")
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

On a linear scale the distribution is a spike near zero with a long right tail; the mean sits far out in the tail. On a log scale it
is a symmetric bell, and the median sits in its middle.

In [ ]:
c = np.linspace(-0.05, 0.15, 400)
peer5 = np.array([-0.02, 0.03, 0.04, 0.05, 0.20])
sq = [np.sum((peer5 - x) ** 2) for x in c]
ab = [np.sum(np.abs(peer5 - x)) for x in c]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(c * 100, sq)
axes[0].axvline(peer5.mean() * 100, color="C3", ls="--", label=f"minimum at the mean, {peer5.mean():.0%}")
axes[0].set_title("Sum of squared errors")
axes[1].plot(c * 100, ab)
axes[1].axvline(np.median(peer5) * 100, color="C2", ls="--", label=f"minimum at the median, {np.median(peer5):.0%}")
axes[1].set_title("Sum of absolute errors")
for ax in axes:
    ax.set_xlabel("forecast c (% return)")
    ax.set_ylabel("loss")
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

The squared loss is a smooth parabola with its minimum at the mean. The absolute loss is piecewise linear, with kinks at each data
point and its minimum at the median.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
bins = np.logspace(-2, 3, 60)
ax.hist(wealth, bins=bins, color="C0", alpha=0.7)
ax.set_xscale("log")
ax.axvline(np.median(wealth), color="C2", ls="--", lw=2, label=f"median {np.median(wealth):.2f}")
ax.axvline(wealth.mean(), color="C3", lw=2, label=f"mean {wealth.mean():.2f}")
ax.axvline(1, color="k", lw=0.8, label="break-even")
ax.set_title("Wealth after 20 years of CHF 1 in each of 1,000 stocks")
ax.set_xlabel("end wealth (CHF, log scale)")
ax.set_ylabel("number of stocks")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

On a log scale end wealth is a bell curve centred on the median; the mean lies well to the right, pulled by the few stocks in the right
tail.

## 7. Exercises

### Basic

1. Compute mean, median and mode of the monthly returns (in %) 1, 2, 2, 3, −4, 2, 5, 1, 2, 30.
2. For a lognormal AuM distribution with median CHF 200 million and log-standard deviation 1.2, compute the mode and the mean.
3. A fund platform advertises "our average client portfolio is worth CHF 1.4 million". The median client portfolio is CHF 180,000.
   **Interpret** both numbers for a prospective client with CHF 250,000.

### Intermediate

4. In the synthetic universe, compute the asset-weighted and equal-weighted average return for the 10% largest and the 90% other funds
   separately. Which group drives the gap?
5. Show numerically that the median of the 1,000 simulated end wealths is a much more stable estimate than the mean: repeat the simulation
   200 times and compare the relative standard deviation of the two estimates.

### Advanced

6. Prove that the median minimizes $E|X - c|$ for a continuous distribution: show that $\frac{d}{dc}E|X - c| = 2F(c) - 1$.
7. Derive the mode, median and mean of the lognormal distribution, and the probability $\Phi(\sigma\sqrt{T}/2)$ that a lognormal wealth after
   $T$ years ends below its mean.

## 8. Solutions / Checks

In [ ]:
# 1.
x1 = np.array([1, 2, 2, 3, -4, 2, 5, 1, 2, 30])
vals, counts = np.unique(x1, return_counts=True)
print(f"1. mean {x1.mean():.1f}%, median {np.median(x1):.1f}%, mode {vals[np.argmax(counts)]}%")

# 2.
m2, s2 = np.log(200), 1.2
print(f"2. mode {np.exp(m2 - s2**2):,.1f}, mean {np.exp(m2 + s2**2 / 2):,.1f} CHF million")

# 4.
big = funds.aum_chf_m >= funds.aum_chf_m.quantile(0.9)
for name, mask in [("largest 10%", big), ("other 90%", ~big)]:
    sub = funds[mask]
    print(f"4. {name}: equal-weighted {sub['return'].mean():.3%}, asset-weighted "
          f"{np.average(sub['return'], weights=sub.aum_chf_m):.3%}, share of AuM {sub.aum_chf_m.sum() / funds.aum_chf_m.sum():.1%}")

# 5.
means, medians = [], []
for _ in range(200):
    w = np.exp(rng.normal(mu_log * years, vol * np.sqrt(years), n_stocks))
    means.append(w.mean())
    medians.append(np.median(w))
print(f"5. relative std of the mean {np.std(means) / np.mean(means):.1%}, of the median {np.std(medians) / np.mean(medians):.1%}")

**1.** The mode (2%) and median (2%) describe the typical month; the mean (4.4%) is pulled up by one 30% month.

**3.** The mean is pulled up by a few very large portfolios; the median says half the clients have less than CHF 180,000. A client with
CHF 250,000 is above the typical client but far below the average, and should compare services and fees with what typical, not average,
clients get.

**4.** Both parts contribute. Within each group the asset-weighted average is about half a point below the equal-weighted one (the size
effect works inside the groups too), and the largest 10%, with 58% of the assets, return about 1.3 points less than the rest. Since they
dominate the asset weights, the universe's asset-weighted return is pulled towards their lower number.

**6.** $E|X - c| = \int_{-\infty}^{c}(c - x)f(x)\,dx + \int_c^{\infty}(x - c)f(x)\,dx$. Differentiating (Leibniz rule; the boundary terms
vanish) gives $\int_{-\infty}^c f - \int_c^\infty f = F(c) - (1 - F(c)) = 2F(c) - 1$, zero at $F(c) = \tfrac12$; the second derivative $2f(c)$
is positive.

**7.** The median of $e^Y$ is $e^{\text{median}(Y)} = e^\mu$ (monotone transformation). The mean is $E[e^Y] = e^{\mu + \sigma^2/2}$ (normal
moment generating function). The density of $X$ is $f(x) = \frac{1}{x\sigma\sqrt{2\pi}}e^{-(\ln x - \mu)^2/(2\sigma^2)}$; setting the derivative
of $\ln f = -\ln x - (\ln x - \mu)^2/(2\sigma^2) + \text{const}$ to zero gives $\ln x = \mu - \sigma^2$. For wealth after $T$ years, $Y \sim N(\mu T, \sigma^2 T)$
and $P(X < \text{mean}) = P\big(Y < \mu T + \sigma^2 T/2\big) = \Phi(\sigma\sqrt{T}/2)$.

## 9. Conclusion

- The mean minimizes squared error, the median absolute error, the mode 0–1 error; the median is robust to outliers.
- Right-skewed data (AuM, wealth, long-horizon stock returns) have mode < median < mean; for lognormal data mean/median $= e^{\sigma^2/2}$.
- Equal-weighted and asset-weighted averages answer different questions; state which one is reported.
- Over long horizons most stocks end below the average stock, because wealth is lognormal and skewed.

Next: [02/02 Volatility and the units of risk](./02_volatility_and_the_units_of_risk.ipynb).